In [2]:
import xarray as xr
import os
path = "/cw3e/mead/projects/cwp140/data/preprocessed/mclimate_2.0/ivt/concat_LT/"
fname = os.path.join(path, "mclimate_ivt_all_leadtimes.nc")
ds = xr.open_dataset(fname)
ds = ds.sel(doy=42)
ds

<xarray.Dataset> Size: 196MB
Dimensions:          (lead_time: 28, quantile: 13, latitude: 281, longitude: 479)
Coordinates:
  * latitude         (latitude) float64 2kB 70.0 69.75 69.5 ... 0.5 0.25 0.0
  * longitude        (longitude) float64 4kB -179.5 -179.2 ... -60.25 -60.0
  * quantile         (quantile) float64 104B 0.0 0.75 0.9 0.91 ... 0.98 0.99 1.0
    doy              int64 8B 42
  * lead_time        (lead_time) int64 224B -9223372036854775806 ... -9223372...
Data variables:
    ivt_percentiles  (lead_time, quantile, latitude, longitude) float32 196MB ...

In [4]:
print(ds.ivt_percentiles.data)

[[[[nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   ...
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]]

  [[nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   ...
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]]

  [[nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   ...
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]]

  ...

  [[nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   ...
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]]

  [[nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   ...
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan na

In [4]:
import xarray as xr
import glob
import os

from config import LEAD_TIME_BLOCKS

path = "/cw3e/mead/projects/cwp140/data/preprocessed/mclimate_2.0/ivt"


for lt in LEAD_TIME_BLOCKS[0:1]:
    print(f"\nConcatenating {lt}")

    files = sorted(glob.glob(
        os.path.join(path, f"mclimate_ivt_{lt}_DOY_*.nc")
    ))

    if not files:
        print("  No files found, skipping")
        continue

    print(f"  {len(files)} files")

    ds = xr.open_mfdataset(
    files,
    combine="nested",
    concat_dim="doy",
    parallel=True,
    chunks={
        "doy": 20,
        "lead_time": 1,
        "quantile": 13,
        "latitude": 141,
        "longitude": 240
        }
    )
    
    ds = ds.transpose("doy", "lead_time", "quantile", "latitude", "longitude")
    print(ds)
    
    out_name = os.path.join( path, f"mclimate_ivt_{lt}.nc" ) 
    tmp = out_name + ".tmp"
    
    encoding = {
        "ivt_percentiles": {
            "zlib": True,
            "complevel": 4,
            "dtype": "float32"
        }
    }
    
    delayed = ds.to_netcdf(
        tmp,
        format="NETCDF4",
        engine="netcdf4",
        encoding=encoding,
        compute=False
    )
    
    delayed.compute()

    os.replace(tmp, out_name)
    ds.close()

    print(f"  Saved → {out_name}")



Concatenating LT_006_012_018
  37 files
<xarray.Dataset> Size: 8GB
Dimensions:          (lead_time: 3, doy: 365, quantile: 13, latitude: 281,
                      longitude: 479)
Coordinates:
  * latitude         (latitude) float64 2kB 70.0 69.75 69.5 ... 0.5 0.25 0.0
  * longitude        (longitude) float64 4kB -179.5 -179.2 ... -60.25 -60.0
  * quantile         (quantile) float64 104B 0.0 0.75 0.9 0.91 ... 0.98 0.99 1.0
  * doy              (doy) int64 3kB 1 2 3 4 5 6 7 ... 360 361 362 363 364 365
  * lead_time        (lead_time) int64 24B 6 12 18
Data variables:
    ivt_percentiles  (doy, lead_time, quantile, latitude, longitude) float32 8GB dask.array<chunksize=(10, 1, 13, 141, 240), meta=np.ndarray>
  Saved → /cw3e/mead/projects/cwp140/data/preprocessed/mclimate_2.0/ivt/mclimate_ivt_LT_006_012_018.nc


In [1]:
## import libraries
import os, sys
import yaml
import xarray as xr
import glob
import pandas as pd

path_to_data = '/expanse/nfs/cw3e/cwp140/'     # project data -- read only
varname = 'freezing_level'

## create list of months and days to iterate through
dates = pd.date_range('2023-12-10', '2023-12-31', freq='1D') ## final dates for MClimate
month_lst = dates.strftime("%-m") # month array
day_lst = dates.strftime("%-d") # day array

for i, (mon, day) in enumerate(zip(month_lst[1:], day_lst[1:])):
    print(mon, day)
    ## get filenames
    fname_pattern = os.path.join(path_to_data, 'preprocessed/mclimate/GEFSv12_reforecast_mclimate_{2}_{0}{1}_*hr-lead.nc').format(mon, day, varname)
    filenames = glob.glob(fname_pattern)
    
    print(len(filenames))
    ds = xr.open_mfdataset(filenames)

    fname = path_to_data + 'preprocessed/ivt_mclimate/GEFSv12_reforecast_mclimate_ivt_{0}{1}.nc'.format(mon.zfill(2), day.zfill(2))
    print(fname)
    ds.to_netcdf(path=fname, mode = 'w', format='NETCDF4')

12 11
40
/expanse/nfs/cw3e/cwp140/preprocessed/ivt_mclimate/GEFSv12_reforecast_mclimate_ivt_1211.nc
12 12
40
/expanse/nfs/cw3e/cwp140/preprocessed/ivt_mclimate/GEFSv12_reforecast_mclimate_ivt_1212.nc
12 13
40
/expanse/nfs/cw3e/cwp140/preprocessed/ivt_mclimate/GEFSv12_reforecast_mclimate_ivt_1213.nc
12 14
40
/expanse/nfs/cw3e/cwp140/preprocessed/ivt_mclimate/GEFSv12_reforecast_mclimate_ivt_1214.nc
12 15
40
/expanse/nfs/cw3e/cwp140/preprocessed/ivt_mclimate/GEFSv12_reforecast_mclimate_ivt_1215.nc
12 16
40
/expanse/nfs/cw3e/cwp140/preprocessed/ivt_mclimate/GEFSv12_reforecast_mclimate_ivt_1216.nc
12 17
40
/expanse/nfs/cw3e/cwp140/preprocessed/ivt_mclimate/GEFSv12_reforecast_mclimate_ivt_1217.nc
12 18
40
/expanse/nfs/cw3e/cwp140/preprocessed/ivt_mclimate/GEFSv12_reforecast_mclimate_ivt_1218.nc
12 19
40
/expanse/nfs/cw3e/cwp140/preprocessed/ivt_mclimate/GEFSv12_reforecast_mclimate_ivt_1219.nc
12 20
40
/expanse/nfs/cw3e/cwp140/preprocessed/ivt_mclimate/GEFSv12_reforecast_mclimate_ivt_1220.nc
